# 00 · Descarga de datos

Este cuaderno **descarga y prepara** los datasets que usaremos en toda la Sesión 1. Se ejecuta una sola vez: a partir de aquí los notebooks 01–03 leen directamente desde `data/raw/`.

## Fuentes

| Variable | Fuente | Estado |
| --- | --- | --- |
| Caudal histórico del Genil (Pinos-Genil 5020, 1913–2021) | CEDEX · Anuario de Aforos | descarga programática vía HTTP |
| Lluvia mensual Iznájar (ref_evap=5001) | CEDEX · Anuario de Aforos | descarga programática vía HTTP |
| Cota piezométrica `PZ0267014` (cuenca Duero) | CHD · Red oficial de piezometría | Excel local |
| **(Opcional)** Caudal reciente Genil-Tocón / lluvia A20_202 | SAIH Guadalquivir | descarga **manual** (ver al final) |


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from cst import datos as ud

plt.rcParams.update({'figure.figsize': (10, 3.5), 'axes.grid': True, 'grid.alpha': 0.3})

## 1 · CEDEX — Anuario de Aforos

El [Anuario de Aforos](https://ceh.cedex.es/anuarioaforos/) del CEDEX-CEH publica los registros oficiales de la red ROEA (Red Oficial de Estaciones de Aforo). Cada demarcación tiene una página con enlaces a varios CSV crudos. Para el Guadalquivir:

- `afliq.csv` — caudales medios diarios.
- `estaf.csv` — catálogo de estaciones (coordenadas, código SAIH equivalente…).
- `evap.csv` — datos mensuales de evaporación y **precipitación** en estaciones evaporimétricas (en embalses).

La URL base es `https://ceh-flumen64.cedex.es/anuarioaforos/anuario-2020-2021/{CUENCA}/{archivo}`. Cobertura: hasta el año hidrológico **2020-21**.

### Detalle técnico interesante

El servidor IIS del CEDEX negocia un cifrado TLS (`AES256-GCM-SHA384` puro, sin ECDHE) que OpenSSL 3 deshabilita por defecto. Por eso `requests` lanza `ConnectionResetError` y `curl` no. La solución en `cst.datos` es un `HTTPAdapter` con `SECLEVEL=1`. Es un caso real de "fricción" que todo data scientist se encontrará tarde o temprano con servidores institucionales antiguos.

In [ ]:
# Descarga (idempotente — usa caché en data/raw/anuario_aforos/)
ruta_afliq = ud.descargar_anuario_csv('afliq.csv')
ruta_estaf = ud.descargar_anuario_csv('estaf.csv')
ruta_evap  = ud.descargar_anuario_csv('evap.csv')

for r in (ruta_afliq, ruta_estaf, ruta_evap):
    print(f'{r.name:25s}  {r.stat().st_size/1e6:6.2f} MB')

### Catálogo de estaciones — encontrar la equivalencia SAIH ↔ ROEA

In [ ]:
est = ud.cargar_anuario_estaciones()
print(f'Total de estaciones en Guadalquivir: {len(est)}')

# Estaciones con cualquier mención al Genil
est_genil = est[est['lugar'].str.contains('GENIL', na=False)]
est_genil[['indroea', 'lugar', 'cod_saih', 'alti', 'longwgs84', 'latwgs84']]

**Lectura:** la SAIH `A20_GENIL_TOCON` (estación telemetrada de la CHG) no aparece en la ROEA — sólo `5020 PINOS-GENIL` (SAIH `A23`) y `5047 PUENTE-GENIL` (cerrada en 2005). Para el curso usaremos **Pinos-Genil 5020** como serie principal: tiene 50+ años de datos diarios reproducibles vía API. Si más adelante descargamos manualmente el SAIH de Genil-Tocón, las utilidades del módulo lo detectarán automáticamente.

In [ ]:
# Caudal diario en Pinos-Genil
caudal = ud.cargar_caudal_genil()  # ROEA 5020
print(ud.resumen(caudal))

fig, ax = plt.subplots()
caudal.loc['1975':'2020'].plot(ax=ax, color='#1f6f8b', lw=0.4)
ax.set_ylabel('Caudal (m³/s)'); ax.set_title('Pinos-Genil (ROEA 5020) — diario')
plt.tight_layout()

In [ ]:
# Lluvia mensual en Iznájar (la pluvio-evaporimétrica más cercana al Genil bajo)
lluvia_mes = ud.cargar_lluvia_genil()  # ref_evap=5001 = embalse Iznájar
print(ud.resumen(lluvia_mes))

fig, ax = plt.subplots()
lluvia_mes.plot(ax=ax, color='#2563eb', lw=0.8)
ax.set_ylabel('Precipitación mensual (mm)'); ax.set_title('Iznájar (ROEA ref_evap=5001)')
plt.tight_layout()

## 2 · CHD — Piezometría

Excel de la Red Oficial de Seguimiento Piezométrico de la **CH del Duero** (`piezometria_chd_2024-12.xlsx`). El fichero ya está en `data/raw/`; lo descargas desde:

- [MITECO · Red de seguimiento piezométrico](https://www.miteco.gob.es/es/agua/temas/evaluacion-de-los-recursos-hidricos/red-oficial-seguimiento/red-seguimiento-piezometrico.html)

Trabajaremos con el pozo `PZ0267014` (provincia de Valladolid, masa de agua del Duero medio). Frecuencia: mediciones manuales, mensuales o irregulares.

In [ ]:
piezo = ud.cargar_piezometria('PZ0267014')
print(ud.resumen(piezo))

fig, ax = plt.subplots()
ax.plot(piezo.index, piezo.values, marker='o', ms=3, lw=0.8, color='#0d9488')
ax.set_ylabel('Cota piezométrica (m s.n.m.)')
ax.set_title('PZ0267014 — registro completo')
plt.tight_layout()

## 3 · (Opcional) SAIH Guadalquivir — datos recientes

Para análisis de eventos recientes (p. ej. la crecida de febrero 2026 mencionada en el currículo), el Anuario de Aforos llega tarde — sólo publica hasta el año hidrológico 2020-21. La fuente actualizada es el **SAIH** de la Confederación Hidrográfica del Guadalquivir:

- Página: [Datos Históricos (SAIH-CHG)](https://www.chguadalquivir.es/saih/DatosHistoricos.aspx).
- No tiene API ni endpoint de descarga directa: es un *WebForm* ASP.NET con `ViewState`. Cualquier scraper se rompe en la siguiente actualización del IIS.

### Cómo descargarlo a mano

1. Entra en la página y despliega el árbol *Sistemas Generales → Genil* hasta el cluster **A20 (Genil-Tocón)**.
2. En el desplegable **Dato**, añade tanto **Caudal** (`A20_211_X`) como **Precipitación** (`A20_202`) — ambas series se exportan en el mismo fichero.
3. Fija el periodo (p. ej. 2018-01-01 → hoy).
4. En **Formato** elige **Excel** y pulsa **Visualizar** → **Exportar**.
5. Guarda el archivo como `data/raw/saih_chg/HistSAIH.xlsx` (crea el subdirectorio si no existe).

El fichero tiene dos hojas: `Info` (metadatos) y `Datos` (FECHA + caudal horario + lluvia horaria, con un bloque *Estadísticas* al final que las funciones del módulo descartan automáticamente).

Si el Excel está presente, `cargar_caudal_genil` y `cargar_lluvia_genil` lo prefieren al ROEA, resampleando a media diaria (caudal) y suma diaria (lluvia):

In [ ]:
print('SAIH Excel disponible:', ud.RUTA_SAIH_XLSX.exists())
if ud.RUTA_SAIH_XLSX.exists():
    print('  →', ud.RUTA_SAIH_XLSX.relative_to(ud.RUTA_RAIZ))

## 4 · Resumen

A partir de este punto, los notebooks `01_*`, `02_*`, `03_*` cargan los datos con:

```python
from cst import datos as ud
caudal = ud.cargar_caudal_genil()
lluvia = ud.cargar_lluvia_genil()
piezo  = ud.cargar_piezometria()
```

Y todo está listo.